# Step 3b - Chronicling America (Library of Congress)

This notebook queries the Library of Congress **loc.gov API** (the new
Chronicling America endpoint at `www.loc.gov/collections/chronicling-america/`)
for newspaper pages mentioning each person in our railroad-ties corpus.

> **Note:** The legacy `chroniclingamerica.loc.gov` API was retired in 2025.
> All queries now go through the loc.gov collections API.

## Strategy

We run **targeted queries** per person, then fall back to a
**broad query** only if the targeted ones return few results:

1. **Obituary query** (no state filter - obituaries appeared nationwide):
   - `"Name" died` - highest signal for obituaries/death notices.

2. **Death-window query** (no state filter, only when death year is known):
   - `"Name"` as phrase search within Â±1 year of death - catches obituaries
     that use unexpected phrasing.

3. **Financial queries** (state-filtered to known states):
   - `"Name" railroad` and `"Name" director bank`
   - Results are **proximity-filtered**: after downloading OCR, we only keep
     pages where a financial keyword appears within 300 chars of the person's
     name. This eliminates the vast majority of false positives from ads and
     market reports elsewhere on the page.

4. **Broad fallback:** `"Name"` as phrase search, used only when queries 1-3
   return fewer than 5 results combined. Results are post-filtered on the
   full OCR text against the keyword sets before storing.

## Filtering

After downloading full-page OCR (via the `word_coordinates_url` text service)
we verify keyword presence before storing, so false positives from the search
stage are caught. Financial query results use a stricter 300-char proximity
filter around the person's name.

## Rate limits

The Library of Congress does not publish hard rate limits but is sensitive to
bulk scraping. We use 1.5 s delay between page downloads and 3 s between
persons. Progress is persisted to the database so reruns are safe.
Expect this step to run over multiple sessions.

In [1]:
import sys, json, time, re
from pathlib import Path
import requests
from tqdm import tqdm

sys.path.insert(0, str(Path('.')))
from db import (
    get_connection, set_progress, pending_persons,
    log_source, store_text,
)

conn = get_connection()
STEP = '3b_chronicling_america'
CA_BASE = "https://www.loc.gov/collections/chronicling-america/"

## Configuration

In [2]:
# --- Tunable parameters ---
MAX_PAGES_PER_PERSON = 200     # max CA pages to download per person
BROAD_FALLBACK_THRESHOLD = 4  # use broad query only if targeted queries return < this many
CONTEXT_CHARS = 500           # chars on each side of name match to store
DOWNLOAD_FULL_OCR = True      # False = log sources only (faster first pass)
DELAY_BETWEEN_PERSONS = 2
DELAY_BETWEEN_PAGES = 1
DELAY_BETWEEN_QUERIES = 1
ROWS_PER_PAGE = 50            # CA API results per request (max ~100)
MAX_API_PAGES = 2             # max pagination depth per query
DRY_RUN = False

## Keyword sets

In [3]:
OBITUARY_KEYWORDS = {
    'died', 'death', 'deceased', 'funeral', 'estate', 'will of',
    'probate', 'surviving', 'obituary', 'obit', 'demise',
    'passed away', 'late of', 'in memoriam', 'years of age',
}

FINANCIAL_KEYWORDS = {
    'railroad', 'railway', 'rail road', 'stock', 'director',
    'investment', 'president of', 'treasurer', 'board of directors',
    'vice president', 'corporation', 'stockholder', 'shares',
    'capital stock', 'bank', 'banking', 'land grant', 'bonds',
    'debenture', 'receiver', 'receivership', 'trustee',
    'superintendent', 'secretary of',
}

# Terms that confirm the deceased was a newspaperman - used to verify
# that an obituary found via unquoted/ambiguous name search is actually
# about our target person (all subjects are newspaper owners/editors).
NEWSPAPER_KEYWORDS = {
    'editor', 'publisher', 'proprietor', 'newspaper', 'paper',
    'journal', 'gazette', 'tribune', 'press', 'daily', 'weekly',
    'writer', 'journalist', 'correspondent', 'reporter',
    'editorial', 'printing', 'printer', 'typographer',
    'founded the', 'edited the', 'published the', 'owner of the',
}

ALL_KEYWORDS = OBITUARY_KEYWORDS | FINANCIAL_KEYWORDS


def get_matching_keywords(text, keywords=ALL_KEYWORDS):
    """Return set of keywords found in text (case-insensitive)."""
    t = text.lower()
    return {kw for kw in keywords if kw in t}


def classify_keywords(kws):
    """Classify a set of matched keywords into categories."""
    return {
        'is_obit': bool(kws & OBITUARY_KEYWORDS),
        'is_financial': bool(kws & FINANCIAL_KEYWORDS),
    }

## API functions

In [4]:
_TAG_RE = re.compile(r'\[\[/?tag\]\]')


def search_ca(query, date_start="1865-01-01", date_end="1910-12-31",
              state=None, count=50, page=1, ops="PHRASE"):
    """
    Low-level Chronicling America search via the loc.gov API.
    Returns (items_list, total_items).
    """
    params = {
        "dl": "page",
        "searchType": "advanced",
        "ops": ops,
        "qs": query,
        "start_date": date_start,
        "end_date": date_end,
        "c": count,
        "sp": page,
        "fo": "json",
    }
    if state:
        params["fa"] = f"location_state:{state.lower()}"

    try:
        resp = requests.get(
            CA_BASE, params=params,
            headers={"User-Agent": "RailroadTiesPipeline/1.0"},
            timeout=45,
        )
        resp.raise_for_status()
        data = resp.json()
        items = data.get("results", [])
        total = data.get("pagination", {}).get("total", 0)
        return items, total
    except requests.exceptions.HTTPError as e:
        if hasattr(e, 'response') and e.response is not None:
            if e.response.status_code == 429:
                print("  Rate limited - waiting 60 s")
                time.sleep(60)
                return search_ca(query, date_start, date_end,
                                 state, count, page, ops)
        print(f"  HTTP error: {e}")
        return [], 0
    except Exception as e:
        print(f"  Search error: {e}")
        return [], 0


def search_ca_paginated(query, date_start, date_end, state=None, ops="PHRASE"):
    """
    Search CA with pagination up to MAX_API_PAGES.
    Returns deduplicated list of item dicts and count.
    """
    all_items = {}
    for pg in range(1, MAX_API_PAGES + 1):
        items, total = search_ca(
            query, date_start, date_end,
            state=state, count=ROWS_PER_PAGE, page=pg, ops=ops,
        )
        for item in items:
            item_id = item.get("id", "")
            if item_id and item_id not in all_items:
                all_items[item_id] = item
        if not items or len(all_items) >= total:
            break
        time.sleep(DELAY_BETWEEN_QUERIES)
    return list(all_items.values()), len(all_items)


def fetch_page_ocr(result_item):
    """
    Fetch full OCR text for a CA search result item.
    Uses the word_coordinates_url from the search result.
    Returns text string or None.
    """
    wc_url = result_item.get("word_coordinates_url", "")
    if not wc_url:
        return None

    try:
        resp = requests.get(
            wc_url + "&full_text=1", timeout=30,
            headers={"User-Agent": "RailroadTiesPipeline/1.0"},
        )
        resp.raise_for_status()
        data = resp.json()
        # Response is {segment_path: {"full_text": "..."}}
        for key, val in data.items():
            text = val.get("full_text", "")
            if text:
                # Strip [[tag]]...[[/tag]] highlighting from search terms
                return _TAG_RE.sub('', text)
        return None
    except Exception as e:
        return None

## Name matching and passage extraction

In [5]:
def build_name_patterns(canonical_name):
    """
    Build a list of regex patterns for finding a person's name in OCR text.
    Handles common OCR substitutions (rnâ†’m, lâ†’1, etc.).
    Returns list of compiled regex patterns, ordered longest-first.
    """
    names = [canonical_name]

    # Also generate last-name-only pattern
    parts = canonical_name.split()
    if len(parts) >= 2:
        names.append(parts[-1])  # last name as fallback

    patterns = []
    for name in names:
        # Build a fuzzy regex: allow common OCR substitutions
        chars = []
        i = 0
        while i < len(name):
            ch = name[i]
            if ch.lower() == 'm':
                chars.append('(?:m|rn)')    # m â†” rn
            elif ch.lower() == 'r' and i + 1 < len(name) and name[i + 1].lower() == 'n':
                chars.append('(?:rn|m)')    # rn â†” m
                i += 1  # skip the 'n'
            elif ch.lower() == 'l':
                chars.append('[l1|]')       # l â†” 1 or |
            elif ch.lower() == 'e':
                chars.append('[ec]')        # e/c confusion
            elif ch == ' ':
                chars.append(r'[\s.,]+')    # flexible whitespace/punctuation
            else:
                chars.append(re.escape(ch))
            i += 1
        pattern = ''.join(chars)
        patterns.append(re.compile(pattern, re.IGNORECASE))

    # Sort: prefer longer (more specific) patterns first
    patterns.sort(key=lambda p: len(p.pattern), reverse=True)
    return patterns


def find_name_in_text(text, patterns):
    """
    Find all match positions for any name pattern in text.
    Returns list of (start, end) tuples, deduplicated by proximity.
    """
    matches = []
    for pat in patterns:
        for m in pat.finditer(text):
            matches.append((m.start(), m.end()))

    if not matches:
        return []

    # Deduplicate overlapping matches
    matches.sort()
    deduped = [matches[0]]
    for s, e in matches[1:]:
        if s >= deduped[-1][1]:  # no overlap
            deduped.append((s, e))
    return deduped


def extract_passages(text, match_positions, context_chars=CONTEXT_CHARS):
    """
    Extract passage text around each name match.
    Returns list of passage strings.
    """
    passages = []
    for start, end in match_positions:
        ctx_start = max(0, start - context_chars)
        ctx_end = min(len(text), end + context_chars)
        passages.append(text[ctx_start:ctx_end])
    return passages


PROXIMITY_WINDOW = 300  # chars on each side of name match for proximity filter


def keywords_near_name(text, match_positions, keywords, window=PROXIMITY_WINDOW):
    """
    Check whether any keyword appears within `window` chars of a name match.
    Returns the set of keywords found near the name, or empty set if none.
    """
    text_lower = text.lower()
    found = set()
    for start, end in match_positions:
        win_start = max(0, start - window)
        win_end = min(len(text), end + window)
        snippet = text_lower[win_start:win_end]
        for kw in keywords:
            if kw in snippet:
                found.add(kw)
    return found


def _name_has_initials(canonical_name):
    """
    Check whether the canonical name already contains initials (1-2 char tokens
    like "S.", "T.", "J"). If so, middle-initial extraction should be skipped
    because the name is already abbreviated.
    """
    parts = canonical_name.split()
    # Check non-last-name parts for initials
    for p in parts[:-1]:
        stripped = p.rstrip('.')
        if len(stripped) <= 2 and stripped[0].isupper():
            return True
    return False


def extract_middle_initial(ocr_text, first_name, last_name, match_positions, window=500):
    """
    Search OCR text near verified name matches for the person's name with a
    middle initial or middle name inserted. Only searches within `window` chars
    of each name match position to avoid picking up different people elsewhere
    on the page.

    Returns the middle component (e.g. "H." or "Henry") or None.
    """
    pattern = (
        re.escape(first_name)
        + r'\s+([A-Z]\.?|[A-Z][a-z]+\.?)\s+'
        + re.escape(last_name)
    )
    pat = re.compile(pattern)
    for start, end in match_positions:
        win_start = max(0, start - window)
        win_end = min(len(ocr_text), end + window)
        snippet = ocr_text[win_start:win_end]
        m = pat.search(snippet)
        if m:
            return m.group(1)
    return None


def verify_newspaperman_obituary(ocr_text, match_positions, window=500):
    """
    Check whether an obituary passage mentions newspaper-related terms
    near the person's name, confirming the deceased was a newspaperman.
    Returns (is_verified, matched_newspaper_keywords).
    """
    nearby_news_kws = keywords_near_name(
        ocr_text, match_positions, NEWSPAPER_KEYWORDS, window=window
    )
    nearby_obit_kws = keywords_near_name(
        ocr_text, match_positions, OBITUARY_KEYWORDS, window=window
    )
    # Verified = has at least one newspaper term AND at least one obituary term near name
    is_verified = bool(nearby_news_kws) and bool(nearby_obit_kws)
    return is_verified, nearby_news_kws, nearby_obit_kws


def extract_death_year_from_date(date_str, last_active_year):
    """
    Extract death year from the publication date of a verified obituary page.
    Only accepts the year if it's plausible: must be >= last_active_year
    (person can't die before they stop being active).
    Returns int or None.
    """
    m = re.match(r'(\d{4})', str(date_str))
    if not m:
        return None
    year = int(m.group(1))
    # Sanity check: death year must be >= last active year
    if last_active_year and year < last_active_year:
        return None
    return year

## Per-person search logic

In [6]:
def search_person(name, states=None, date_start="1865", date_end="1910",
                  death_year=None, last_active_year=None):
    """
    Run targeted + fallback queries for one person.

    Returns dict keyed by item ID:
        {id: {id, url, date, title, state, query_type, raw_item}}
    """
    results = {}  # id -> parsed item
    state_list = states if states else [None]

    # Dates need YYYY-MM-DD for the new API
    ds = f"{date_start}-01-01"
    de = f"{date_end}-12-31"

    # Obituary queries use a wider date window: people can die decades
    # after their last recorded active year.
    obit_last = last_active_year or int(date_end)
    obit_de = f"{min(obit_last + 40, 1923)}-12-31"

    quoted_name = f'"{name}"'

    # ---- OBITUARY QUERY - quoted name, no state filter ----

    items, _ = search_ca_paginated(
        f'{quoted_name} died', ds, obit_de, state=None, ops="AND",
    )
    for item in items:
        iid = item.get("id", "")
        if iid and iid not in results:
            results[iid] = _parse_item(item, query_type='obit_died')
    time.sleep(DELAY_BETWEEN_QUERIES)

    # ---- OBITUARY STATE QUERY - unquoted name, state-filtered ----
    # Catches middle-initial variants (e.g. "Isaac H. Bowman" when
    # canonical is "Isaac Bowman"). Verified at OCR download time
    # by requiring newspaper-related terms near the name.

    for state in state_list[:3]:
        if state is None:
            continue
        items, _ = search_ca_paginated(
            f'{name} died', ds, obit_de, state=state, ops="AND",
        )
        for item in items:
            iid = item.get("id", "")
            if iid and iid not in results:
                results[iid] = _parse_item(item, query_type='obit_state')
        time.sleep(DELAY_BETWEEN_QUERIES)

    # ---- DEATH-WINDOW QUERY (no state filter, Â±1 year of death) ----

    if death_year:
        dw_start = f"{death_year - 1}-01-01"
        dw_end   = f"{min(death_year + 1, 1923)}-12-31"
        items, _ = search_ca_paginated(
            quoted_name, dw_start, dw_end, state=None, ops="PHRASE",
        )
        for item in items:
            iid = item.get("id", "")
            if iid and iid not in results:
                results[iid] = _parse_item(item, query_type='death_window')
        time.sleep(DELAY_BETWEEN_QUERIES)

    # ---- FINANCIAL QUERIES (state-filtered; proximity-filtered at download) ----

    for state in state_list[:3]:
        fin_queries = [
            (f'{quoted_name} railroad',      'financial_rr'),
            (f'{quoted_name} director bank', 'financial_dir'),
        ]
        for query, qtype in fin_queries:
            items, _ = search_ca_paginated(
                query, ds, de, state=state, ops="AND",
            )
            for item in items:
                iid = item.get("id", "")
                if iid and iid not in results:
                    results[iid] = _parse_item(item, query_type=qtype)
            time.sleep(DELAY_BETWEEN_QUERIES)

    # ---- BROAD FALLBACK (phrase search, state-filtered) ----

    if len(results) < BROAD_FALLBACK_THRESHOLD:
        for state in state_list[:3]:
            items, _ = search_ca_paginated(
                quoted_name, ds, de, state=state, ops="PHRASE",
            )
            for item in items:
                iid = item.get("id", "")
                if iid and iid not in results:
                    results[iid] = _parse_item(item, query_type='broad')
            time.sleep(DELAY_BETWEEN_QUERIES)

    return results


def _parse_item(item, query_type=''):
    """Normalize a loc.gov API search result into our internal format."""
    item_id = item.get("id", "")
    url = item.get("url", "") or item_id.replace("http://", "https://")
    states = item.get("location_state", [])
    return {
        "id": item_id,
        "url": url,
        "date": item.get("date", ""),
        "title": item.get("title", ""),
        "state": states,
        "query_type": query_type,
        "raw_item": item,  # keep full item for OCR retrieval
    }

## Main processing loop

In [7]:
todo = pending_persons(conn, STEP)
print(f"{len(todo)} persons pending for step '{STEP}'")
print(f"Config: max {MAX_PAGES_PER_PERSON} pages/person, "
      f"download_ocr={DOWNLOAD_FULL_OCR}, dry_run={DRY_RUN}\n")

505 persons pending for step '3b_chronicling_america'
Config: max 200 pages/person, download_ocr=True, dry_run=False



In [8]:
# State name -> CA API state value
# CA expects full state names in title case.
# This map normalizes common formats from our source data.
_STATE_NORMALIZE = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut',
    'DE': 'Delaware', 'DC': 'District of Columbia', 'FL': 'Florida',
    'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois',
    'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas', 'KY': 'Kentucky',
    'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota',
    'MS': 'Mississippi', 'MO': 'Missouri', 'MT': 'Montana',
    'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire',
    'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania',
    'RI': 'Rhode Island', 'SC': 'South Carolina', 'SD': 'South Dakota',
    'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
    'WI': 'Wisconsin', 'WY': 'Wyoming',
}
# Also accept full uppercase names
_STATE_NORMALIZE.update({v.upper(): v for v in _STATE_NORMALIZE.values()})
# And title case pass-through
_STATE_NORMALIZE.update({v: v for v in _STATE_NORMALIZE.values()})


def normalize_state(s):
    """Convert state string to CA API format. Returns None if unrecognized."""
    s = s.strip()
    return _STATE_NORMALIZE.get(s) or _STATE_NORMALIZE.get(s.upper())

In [ ]:
# API Smoke Test 
# Verifies search logic, applies all filters (proximity, newspaper
# verification), and reports on enrichment discoveries (middle initials,
# death years). Saves a detailed report for LLM review.

import random
from collections import defaultdict

SAMPLE_PER_QUERY_TYPE = 15
FINANCIAL_QUERY_TYPES_TEST = {'financial_rr', 'financial_dir'}


def run_smoke_test_for_person(conn, research_id, label=""):
    person = conn.execute(
        "SELECT research_id, canonical_name, known_states, first_active_year, last_active_year "
        "FROM persons WHERE research_id=?", (research_id,)
    ).fetchone()

    name = person["canonical_name"]
    name_parts = name.split()
    first_name = name_parts[0] if name_parts else ""
    last_name = name_parts[-1] if len(name_parts) >= 2 else ""
    can_extract_middle = (first_name and last_name
                          and not _name_has_initials(name))

    states_raw = json.loads(person["known_states"] or "[]")
    states = [s for s in (normalize_state(s) for s in states_raw) if s]

    death_row = conn.execute(
        "SELECT death_year FROM enrichment WHERE research_id=? AND death_year IS NOT NULL LIMIT 1",
        (research_id,)
    ).fetchone()
    death_year = death_row["death_year"] if death_row else None

    first_yr = person["first_active_year"] or 1865
    last_yr  = person["last_active_year"]  or 1895
    if death_year:
        date_end = str(min(death_year + 5, 1923))
    else:
        date_end = str(min(last_yr + 15, 1923))
    date_start = str(max(first_yr - 5, 1836))

    meta = {
        "label": label, "name": name, "research_id": research_id,
        "states": states, "date_start": date_start, "date_end": date_end,
        "death_year": death_year,
        "last_active_year": last_yr,
        "obit_date_end": str(min(last_yr + 40, 1923)),
        "can_extract_middle": can_extract_middle,
    }

    print(f"Searching: {name} (rid={research_id}), states={states}, "
          f"dates={date_start}-{date_end} (obit to {min(last_yr+40,1923)}), death={death_year}")
    if not can_extract_middle:
        print(f"  (skipping middle-initial extraction: name already has initials)")

    results = search_person(
        name, states=states or None,
        date_start=date_start, date_end=date_end,
        death_year=death_year, last_active_year=last_yr,
    )

    by_type = defaultdict(int)
    for r in results.values():
        by_type[r["query_type"]] += 1
    print(f"  {len(results)} results: {dict(sorted(by_type.items()))}")

    # Sample up to SAMPLE_PER_QUERY_TYPE per query type
    type_counts = defaultdict(int)
    sampled_ids = []
    for iid, r in results.items():
        qt = r["query_type"]
        if type_counts[qt] < SAMPLE_PER_QUERY_TYPE:
            sampled_ids.append(iid)
            type_counts[qt] += 1

    print(f"  Downloading OCR for {len(sampled_ids)} sampled pages...")

    patterns = build_name_patterns(name)
    passages = []
    proximity_filtered = 0
    discovered_middle = None
    discovered_death_year = None

    for i, iid in enumerate(sampled_ids):
        r = results[iid]
        query_type = r["query_type"]
        ocr_text = fetch_page_ocr(r.get("raw_item", {}))

        if not ocr_text:
            passages.append({
                "index": i + 1, "query_type": query_type,
                "date": r.get("date", ""), "title": r.get("title", ""),
                "url": r.get("url", iid),
                "ocr_status": "FAILED", "passage": None,
                "keywords_matched": [], "proximity_filtered": False,
            })
            time.sleep(DELAY_BETWEEN_PAGES)
            continue

        matches = find_name_in_text(ocr_text, patterns)

        # --- Financial proximity filter ---
        if query_type in FINANCIAL_QUERY_TYPES_TEST:
            if not matches:
                passages.append({
                    "index": i + 1, "query_type": query_type,
                    "date": r.get("date", ""), "title": r.get("title", ""),
                    "url": r.get("url", iid),
                    "ocr_status": "OK", "passage": None,
                    "keywords_matched": [], "proximity_filtered": True,
                    "filter_reason": "no name match in OCR",
                    "name_matches_found": 0,
                })
                proximity_filtered += 1
                time.sleep(DELAY_BETWEEN_PAGES)
                continue
            nearby_fin_kws = keywords_near_name(ocr_text, matches, FINANCIAL_KEYWORDS)
            if not nearby_fin_kws:
                page_kws = get_matching_keywords(ocr_text, FINANCIAL_KEYWORDS)
                passages.append({
                    "index": i + 1, "query_type": query_type,
                    "date": r.get("date", ""), "title": r.get("title", ""),
                    "url": r.get("url", iid),
                    "ocr_status": "OK", "passage": None,
                    "keywords_matched": sorted(page_kws),
                    "proximity_filtered": True,
                    "filter_reason": f"financial keywords on page but not within {PROXIMITY_WINDOW} chars of name",
                    "name_matches_found": len(matches),
                })
                proximity_filtered += 1
                time.sleep(DELAY_BETWEEN_PAGES)
                continue
            kws = nearby_fin_kws

        # --- obit_state: newspaper verification filter ---
        elif query_type == 'obit_state':
            if not matches:
                passages.append({
                    "index": i + 1, "query_type": query_type,
                    "date": r.get("date", ""), "title": r.get("title", ""),
                    "url": r.get("url", iid),
                    "ocr_status": "OK", "passage": None,
                    "keywords_matched": [], "proximity_filtered": True,
                    "filter_reason": "no name match in OCR",
                    "name_matches_found": 0,
                })
                proximity_filtered += 1
                time.sleep(DELAY_BETWEEN_PAGES)
                continue
            is_verified, news_kws, obit_kws = verify_newspaperman_obituary(ocr_text, matches)
            if not is_verified:
                page_news = keywords_near_name(ocr_text, matches, NEWSPAPER_KEYWORDS, window=500)
                page_obit = keywords_near_name(ocr_text, matches, OBITUARY_KEYWORDS, window=500)
                reason = "missing "
                if not page_obit:
                    reason += "obituary terms"
                if not page_news:
                    reason += (" + " if "obituary" in reason else "") + "newspaper terms"
                reason += " near name"
                passages.append({
                    "index": i + 1, "query_type": query_type,
                    "date": r.get("date", ""), "title": r.get("title", ""),
                    "url": r.get("url", iid),
                    "ocr_status": "OK", "passage": None,
                    "keywords_matched": sorted(page_news | page_obit),
                    "proximity_filtered": True,
                    "filter_reason": reason,
                    "name_matches_found": len(matches),
                })
                proximity_filtered += 1
                time.sleep(DELAY_BETWEEN_PAGES)
                continue
            kws = obit_kws | news_kws

            # Try enrichment extraction (proximity-limited + initials guard)
            if can_extract_middle and not discovered_middle:
                mid = extract_middle_initial(ocr_text, first_name, last_name, matches)
                if mid:
                    discovered_middle = mid
            if not death_year and not discovered_death_year:
                yr = extract_death_year_from_date(r.get("date", ""), last_yr)
                if yr:
                    discovered_death_year = yr

        # --- Other query types: page-level filter ---
        else:
            kws = get_matching_keywords(ocr_text)
            # Middle initial extraction is only done from obit_state results
            # (state-filtered + newspaperman-verified), not from nationwide
            # obit_died/death_window which may match a different person.
            pass

        if matches:
            extracted = extract_passages(ocr_text, matches)
            passage_text = "\n---\n".join(extracted)
        else:
            passage_text = ocr_text[:2000]

        passages.append({
            "index": i + 1, "query_type": query_type,
            "date": r.get("date", ""), "title": r.get("title", ""),
            "url": r.get("url", iid),
            "ocr_status": "OK", "passage": passage_text,
            "keywords_matched": sorted(kws),
            "name_matches_found": len(matches),
            "proximity_filtered": False,
        })
        time.sleep(DELAY_BETWEEN_PAGES)

    ok = sum(1 for p in passages if p['ocr_status'] == 'OK' and not p['proximity_filtered'])
    print(f"  Done: {ok} kept, {proximity_filtered} filtered, "
          f"{sum(1 for p in passages if p['ocr_status']=='FAILED')} OCR failed")

    if discovered_middle:
        full_variant = f"{first_name} {discovered_middle} {last_name}"
        print(f"  ENRICHMENT: middle initial â†’ {full_variant}")
        meta["discovered_middle"] = full_variant
    if discovered_death_year:
        print(f"  ENRICHMENT: death year â†’ {discovered_death_year}")
        meta["discovered_death_year"] = discovered_death_year

    return results, passages, meta


# â”€â”€ Whitelaw Reid â”€â”€
reid = conn.execute(
    "SELECT research_id FROM persons WHERE canonical_name LIKE '%Whitelaw Reid%' LIMIT 1"
).fetchone()
assert reid, "Whitelaw Reid not found"
reid_results, reid_passages, reid_meta = run_smoke_test_for_person(
    conn, reid["research_id"], label="Whitelaw Reid (benchmark)"
)

BENCHMARKS = [
    ("sn83035565", "1912-12-19", "Benchmark 1"),
    ("sn86061215", "1912-12-17", "Benchmark 2"),
    ("sn84020422", "1912-12-18", "Benchmark 3"),
]
benchmark_pass = 0
for lccn, date, label in BENCHMARKS:
    found = any(lccn in iid and date in iid for iid in reid_results)
    status = "PASS" if found else "FAIL"
    if found:
        benchmark_pass += 1
    print(f"  [{status}] {label}: {lccn}/{date}")
print(f"  Benchmark score: {benchmark_pass}/{len(BENCHMARKS)}")

# â”€â”€ Isaac Bowman (middle initial benchmark) â”€â”€
print()
bowman = conn.execute(
    "SELECT research_id FROM persons WHERE canonical_name LIKE '%Isaac Bowman%' LIMIT 1"
).fetchone()
if bowman:
    bowman_results, bowman_passages, bowman_meta = run_smoke_test_for_person(
        conn, bowman["research_id"], label="Isaac Bowman (middle initial benchmark)"
    )
    # Check if the known obituary was found
    bowman_obit_found = any("sn88056158" in iid and "1903-06-16" in iid for iid in bowman_results)
    print(f"  [{'PASS' if bowman_obit_found else 'FAIL'}] Bowman obit: sn88056158/1903-06-16")
else:
    bowman_results, bowman_passages, bowman_meta = {}, [], {}
    print("  Isaac Bowman not found in persons table")

# â”€â”€ Random person â”€â”€
print()
all_rids = [r["research_id"] for r in conn.execute("SELECT research_id FROM persons").fetchall()]
exclude = {reid["research_id"]}
if bowman:
    exclude.add(bowman["research_id"])
rand_rid = random.choice([r for r in all_rids if r not in exclude])
rand_results, rand_passages, rand_meta = run_smoke_test_for_person(
    conn, rand_rid, label="Random person"
)

# â”€â”€ Write report â”€â”€
report_path = Path("step3b_smoke_test_report.txt")
lines = []

test_sets = [(reid_meta, reid_passages, reid_results)]
if bowman_passages:
    test_sets.append((bowman_meta, bowman_passages, bowman_results))
test_sets.append((rand_meta, rand_passages, rand_results))

for meta, passages, results in test_sets:
    lines.append("=" * 80)
    lines.append(f"PERSON: {meta['name']}  (research_id={meta['research_id']})")
    lines.append(f"  States: {meta['states']}")
    lines.append(f"  Date range: {meta['date_start']} - {meta['date_end']}")
    lines.append(f"  Death year: {meta['death_year']}")
    lines.append(f"  Can extract middle initial: {meta.get('can_extract_middle', 'N/A')}")

    by_type = defaultdict(int)
    for r in results.values():
        by_type[r["query_type"]] += 1
    lines.append(f"  Total search results: {len(results)}")
    lines.append(f"  By query type: {dict(sorted(by_type.items()))}")

    kept = sum(1 for p in passages if not p.get("proximity_filtered") and p["ocr_status"] == "OK")
    filtered = sum(1 for p in passages if p.get("proximity_filtered"))
    lines.append(f"  OCR sampled: {len(passages)} ({SAMPLE_PER_QUERY_TYPE}/type)")
    lines.append(f"  Kept after filters: {kept}  |  Filtered out: {filtered}")

    if meta.get("discovered_middle"):
        lines.append(f"  ENRICHMENT DISCOVERED: middle initial â†’ {meta['discovered_middle']}")
    if meta.get("discovered_death_year"):
        lines.append(f"  ENRICHMENT DISCOVERED: death year â†’ {meta['discovered_death_year']}")
    lines.append("")

    for p in passages:
        lines.append("-" * 70)
        lines.append(f"Result #{p['index']}  |  Query: {p['query_type']}  |  Date: {p['date']}")
        lines.append(f"Title: {p['title']}")
        lines.append(f"URL: {p['url']}")

        if p.get("proximity_filtered"):
            lines.append(f"*** FILTERED - {p.get('filter_reason', 'discarded')} ***")
            lines.append(f"Page-level keywords: {', '.join(p['keywords_matched']) if p['keywords_matched'] else 'none'}")
        else:
            lines.append(f"OCR: {p['ocr_status']}  |  Name matches: {p.get('name_matches_found', 'N/A')}")
            lines.append(f"Keywords: {', '.join(p['keywords_matched']) if p['keywords_matched'] else 'none'}")
            if p.get("passage"):
                lines.append("")
                lines.append(p["passage"])
        lines.append("")

    lines.append("")

report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"\nReport saved to {report_path} ({len(lines)} lines, {report_path.stat().st_size / 1024:.0f} KB)")

Searching: Whitelaw Reid (rid=1), states=['New York'], dates=1868-1917 (obit to 1923), death=1912
  223 results: {'death_window': 18, 'financial_dir': 36, 'financial_rr': 32, 'obit_died': 100, 'obit_state': 37}
  Done: 52 kept, 23 filtered, 0 OCR failed
  [PASS] Benchmark 1: sn83035565/1912-12-19
  [PASS] Benchmark 2: sn86061215/1912-12-17
  [PASS] Benchmark 3: sn84020422/1912-12-18
  Benchmark score: 3/3

Searching: Isaac Bowman (rid=134), states=['Idaho'], dates=1864-1884 (obit to 1909), death=None
  100 results: {'obit_died': 50, 'obit_state': 50}
  Done: 16 kept, 14 filtered, 0 OCR failed
  ENRICHMENT: middle initial â†’ Isaac H. Bowman
  ENRICHMENT: death year â†’ 1903
  [PASS] Bowman obit: sn88056158/1903-06-16

Searching: Margaret Fuller (rid=7), states=['New York'], dates=1864-1923 (obit to 1911), death=1944
  351 results: {'death_window': 100, 'financial_dir': 25, 'financial_rr': 42, 'obit_died': 100, 'obit_state': 84}
  Done: 39 kept, 35 filtered, 1 OCR failed

Report saved t

In [ ]:
FINANCIAL_QUERY_TYPES = {'financial_rr', 'financial_dir'}

dry_run_log = []

for research_id in tqdm(todo if not DRY_RUN else todo[:5]):
    person = conn.execute(
        """SELECT canonical_name, known_states,
                  first_active_year, last_active_year
           FROM persons WHERE research_id = ?""",
        (research_id,),
    ).fetchone()

    name = person["canonical_name"]
    name_parts = name.split()
    first_name = name_parts[0] if name_parts else ""
    last_name = name_parts[-1] if len(name_parts) >= 2 else ""
    can_extract_middle = (first_name and last_name
                          and not _name_has_initials(name))

    # --- Resolve states ---
    states_raw = json.loads(person["known_states"] or "[]")
    states = [s for s in (normalize_state(s) for s in states_raw) if s]

    # --- Date range ---
    death_row = conn.execute(
        """SELECT death_year FROM enrichment
           WHERE research_id = ? AND death_year IS NOT NULL
           LIMIT 1""",
        (research_id,),
    ).fetchone()

    death_year = death_row["death_year"] if death_row else None
    first_yr = person["first_active_year"] or 1865
    last_yr  = person["last_active_year"]  or 1895

    if death_year:
        date_end = str(min(death_year + 5, 1923))
    else:
        date_end = str(min(last_yr + 15, 1923))
    date_start = str(max(first_yr - 5, 1836))

    # --- Search ---
    try:
        ca_results = search_person(
            name, states=states or None,
            date_start=date_start, date_end=date_end,
            death_year=death_year, last_active_year=last_yr,
        )

        if DRY_RUN:
            lines = []
            lines.append(f"{'='*60}")
            lines.append(f"{name} (rid={research_id}) - states={states}, "
                         f"dates={date_start}-{date_end}, death={death_year}")
            lines.append(f"  Results: {len(ca_results)} pages found")
            by_type = {}
            for iid, r in ca_results.items():
                qt = r.get("query_type", "?")
                by_type[qt] = by_type.get(qt, 0) + 1
            for qt, count in sorted(by_type.items()):
                lines.append(f"    {qt}: {count}")
            for iid, r in list(ca_results.items())[:10]:
                lines.append(f"  - [{r.get('query_type','?'):<16}] "
                             f"{r.get('date',''):<12} {r.get('title','')[:60]}")
                lines.append(f"    {r.get('url', iid)}")
            if len(ca_results) > 10:
                lines.append(f"  ... and {len(ca_results) - 10} more")
            lines.append("")
            block = "\n".join(lines)
            print(block)
            dry_run_log.append(block)
            continue

        if not ca_results:
            set_progress(conn, research_id, STEP, 'done', result_count=0)
            time.sleep(DELAY_BETWEEN_PERSONS)
            continue

        # --- Build name patterns for OCR matching ---
        patterns = build_name_patterns(name)

        pages_stored = 0
        discovered_death_year = None
        discovered_middle = None

        for iid, result in list(ca_results.items())[:MAX_PAGES_PER_PERSON]:
            url = result.get("url", iid)
            query_type = result.get("query_type", "")

            # Log the discovered source regardless of download
            source_id = log_source(
                conn, research_id, 'chronicling_america',
                title=result.get("title", ""),
                url=url,
                item_id=iid,
                snippet="",
                relevance_note=(
                    f"date:{result.get('date', '')}; "
                    f"query_type:{query_type}"
                ),
                discovery_step='3b_ca',
            )

            if not DOWNLOAD_FULL_OCR:
                continue

            # --- Fetch full OCR ---
            ocr_text = fetch_page_ocr(result.get("raw_item", {}))
            if not ocr_text:
                time.sleep(DELAY_BETWEEN_PAGES)
                continue

            # --- Find name positions ---
            matches = find_name_in_text(ocr_text, patterns)

            # --- Query-type-specific filtering ---

            if query_type in FINANCIAL_QUERY_TYPES:
                # Financial proximity filter
                if not matches:
                    time.sleep(DELAY_BETWEEN_PAGES)
                    continue
                nearby_fin_kws = keywords_near_name(ocr_text, matches, FINANCIAL_KEYWORDS)
                if not nearby_fin_kws:
                    time.sleep(DELAY_BETWEEN_PAGES)
                    continue
                kws = nearby_fin_kws

            elif query_type == 'obit_state':
                # Unquoted name query - must verify with newspaper terms
                if not matches:
                    time.sleep(DELAY_BETWEEN_PAGES)
                    continue
                is_verified, news_kws, obit_kws = verify_newspaperman_obituary(
                    ocr_text, matches
                )
                if not is_verified:
                    time.sleep(DELAY_BETWEEN_PAGES)
                    continue
                kws = obit_kws | news_kws

                # --- Enrichment: extract middle initial and death year ---
                if can_extract_middle and not discovered_middle:
                    mid = extract_middle_initial(ocr_text, first_name, last_name, matches)
                    if mid:
                        discovered_middle = mid

                if not death_year and not discovered_death_year:
                    yr = extract_death_year_from_date(result.get("date", ""), last_yr)
                    if yr:
                        discovered_death_year = yr

            else:
                # obit_died, death_window, broad: page-level keyword filter
                kws = get_matching_keywords(ocr_text)
                if query_type == 'broad' and not kws:
                    time.sleep(DELAY_BETWEEN_PAGES)
                    continue

                # Middle initial extraction is only done from obit_state results
                # (state-filtered + newspaperman-verified), not from nationwide
                # obit_died/death_window which may match a different person.
                pass

            # --- Extract passages ---
            if matches:
                passages = extract_passages(ocr_text, matches)
                combined_passage = "\n---\n".join(passages)
            else:
                combined_passage = ocr_text[:2000]

            store_text(
                conn, research_id,
                passage_text=combined_passage,
                context_text="",
                source_id=source_id,
                source_type='chronicling_america',
                source_url=url,
                source_title=result.get("title", ""),
                is_keyword_filtered=1 if kws else 0,
                keyword_match=", ".join(sorted(kws)[:10]),
                page_num=result.get("date", ""),
            )
            pages_stored += 1
            time.sleep(DELAY_BETWEEN_PAGES)

        # --- Post-processing: persist discovered enrichments ---

        if discovered_middle:
            # Build the full name with middle initial
            full_variant = f"{first_name} {discovered_middle} {last_name}"
            # Add any extra name parts (suffixes like Jr., III)
            if len(name_parts) > 2:
                # Original might be "John Smith Jr." - insert middle before last
                full_variant = f"{first_name} {discovered_middle} {' '.join(name_parts[1:])}"

            existing = conn.execute(
                "SELECT name_variants FROM persons WHERE research_id=?",
                (research_id,)
            ).fetchone()
            current_variants = json.loads(existing["name_variants"] or "[]") if existing["name_variants"] else []
            if full_variant not in current_variants:
                current_variants.append(full_variant)
                conn.execute(
                    "UPDATE persons SET name_variants=? WHERE research_id=?",
                    (json.dumps(current_variants), research_id)
                )
                conn.commit()
                print(f"  + Discovered middle initial for {name}: {full_variant}")

        if discovered_death_year and not death_year:
            # Store as enrichment from CA obituary
            from db import store_enrichment
            store_enrichment(
                conn, research_id, source='chronicling_america',
                death_year=discovered_death_year,
            )
            print(f"  + Discovered death year for {name}: {discovered_death_year}")

            # Run death-window query now that we have a death year
            quoted_name = f'"{name}"'
            dw_start = f"{discovered_death_year - 1}-01-01"
            dw_end   = f"{min(discovered_death_year + 1, 1923)}-12-31"
            dw_items, _ = search_ca_paginated(
                quoted_name, dw_start, dw_end, state=None, ops="PHRASE",
            )
            dw_new = 0
            for item in dw_items:
                dw_iid = item.get("id", "")
                if dw_iid and dw_iid not in ca_results:
                    dw_result = _parse_item(item, query_type='death_window_late')

                    dw_source_id = log_source(
                        conn, research_id, 'chronicling_america',
                        title=dw_result.get("title", ""),
                        url=dw_result.get("url", dw_iid),
                        item_id=dw_iid, snippet="",
                        relevance_note=f"date:{dw_result.get('date','')}; query_type:death_window_late",
                        discovery_step='3b_ca',
                    )

                    ocr_text = fetch_page_ocr(item)
                    if ocr_text:
                        dw_matches = find_name_in_text(ocr_text, patterns)
                        dw_kws = get_matching_keywords(ocr_text)
                        if dw_matches:
                            dw_passages = extract_passages(ocr_text, dw_matches)
                            combined = "\n---\n".join(dw_passages)
                        else:
                            combined = ocr_text[:2000]
                        store_text(
                            conn, research_id,
                            passage_text=combined, context_text="",
                            source_id=dw_source_id,
                            source_type='chronicling_america',
                            source_url=dw_result.get("url", dw_iid),
                            source_title=dw_result.get("title", ""),
                            is_keyword_filtered=1 if dw_kws else 0,
                            keyword_match=", ".join(sorted(dw_kws)[:10]),
                            page_num=dw_result.get("date", ""),
                        )
                        pages_stored += 1
                        dw_new += 1
                    time.sleep(DELAY_BETWEEN_PAGES)

            if dw_new:
                print(f"    Death-window re-query found {dw_new} additional pages")

        set_progress(conn, research_id, STEP, 'done',
                     result_count=pages_stored)

    except Exception as e:
        print(f"\n  Error for {name} ({research_id}): {e}")
        if not DRY_RUN:
            set_progress(conn, research_id, STEP, 'failed',
                         error_msg=str(e))

    time.sleep(DELAY_BETWEEN_PERSONS)

if DRY_RUN and dry_run_log:
    out_path = Path("step3b_dry_run.txt")
    out_path.write_text("\n".join(dry_run_log), encoding="utf-8")
    print(f"\nDry run lo
    g saved to {out_path} ({len(dry_run_log)} persons)")

  0%|          | 0/505 [00:00<?, ?it/s]


  Error for A. J. Steinman (66): FOREIGN KEY constraint failed


  0%|          | 1/505 [00:15<2:10:24, 15.52s/it]

  + Discovered death year for E. H. Thomas: 1907
    Death-window re-query found 85 additional pages


  0%|          | 2/505 [15:04<73:58:44, 529.47s/it]

...

  5%|▍         | 25/505 [4:20:06<88:35:51, 664.48s/it]

  + Discovered death year for Thomas O. Ward: 1898
    Death-window re-query found 100 additional pages


  5%|▌         | 27/505 [4:35:48<69:56:47, 526.79s/it]

  + Discovered death year for S. T. Conway: 1911
  Search error: HTTPSConnectionPool(host='www.loc.gov', port=443): Read timed out. (read timeout=45)


  6%|▌         | 28/505 [4:45:18<71:31:52, 539.86s/it]

## Summary

In [ ]:
print("=== Step 3b Summary ===\n")

total_stored = conn.execute(
    "SELECT COUNT(*) as n FROM texts_downloaded WHERE source_type = 'chronicling_america'"
).fetchone()["n"]

obit_total = conn.execute("""
    SELECT COUNT(*) as n FROM texts_downloaded
    WHERE source_type = 'chronicling_america'
      AND (keyword_match LIKE '%died%'
           OR keyword_match LIKE '%death%'
           OR keyword_match LIKE '%obituary%'
           OR keyword_match LIKE '%probate%')
""").fetchone()["n"]

rr_total = conn.execute("""
    SELECT COUNT(*) as n FROM texts_downloaded
    WHERE source_type = 'chronicling_america'
      AND (keyword_match LIKE '%railroad%'
           OR keyword_match LIKE '%railway%'
           OR keyword_match LIKE '%director%')
""").fetchone()["n"]

persons_with_results = conn.execute("""
    SELECT COUNT(DISTINCT research_id) as n FROM texts_downloaded
    WHERE source_type = 'chronicling_america'
""").fetchone()["n"]

persons_total = conn.execute("SELECT COUNT(*) as n FROM persons").fetchone()["n"]

print(f"Total CA pages stored:     {total_stored}")
print(f"  With obituary keywords:  {obit_total}")
print(f"  With financial keywords: {rr_total}")
print(f"Persons with â‰¥1 CA result: {persons_with_results} / {persons_total}")

print(f"\nTop 20 persons by CA passages:")
top = conn.execute("""
    SELECT p.canonical_name, COUNT(*) as pages
    FROM texts_downloaded t
    JOIN persons p ON t.research_id = p.research_id
    WHERE t.source_type = 'chronicling_america'
    GROUP BY t.research_id
    ORDER BY pages DESC
    LIMIT 20
""").fetchall()
for r in top:
    print(f"  {r['canonical_name']:<35} {r['pages']:>4} pages")

print(f"\nPersons with 0 results (first 20):")
zeros = conn.execute("""
    SELECT p.canonical_name
    FROM persons p
    JOIN pipeline_progress pr ON p.research_id = pr.research_id
    WHERE pr.step = ? AND pr.status = 'done' AND pr.result_count = 0
    LIMIT 20
""", (STEP,)).fetchall()
for r in zeros:
    print(f"  {r['canonical_name']}")